In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime
import polars as pl

from fart.constants import DATETIME, MAGNITUDE
from fart.features.calculate_magnitude import calculate_magnitude
from fart.features.calculate_technical_indicators import calculate_technical_indicators
from fart.features.calculate_trade_returns import calculate_trade_returns
from fart.features.parse_timestamp_to_datetime import parse_timestamp_to_datetime
from fart.utils import get_project_root, get_data_filepath
from fart.visualization.candlestick_chart import plot_candlestick_chart
from fart.visualization.magnitude import plot_magnitude
from fart.visualization.missing_value_heatmap import plot_missing_value_heatmap
from fart.visualization.plot_styles import apply_plot_styles
from fart.visualization.trade_returns import plot_trade_returns

apply_plot_styles()

In [ ]:
assets_dir = get_project_root() / "assets"
candle_filepath = get_data_filepath(assets_dir, "BTC-EUR", "1d")

df = pl.read_csv(candle_filepath)

In [ ]:
plot_missing_value_heatmap(df)

In [ ]:
df = parse_timestamp_to_datetime(df)
# Calculate technical indicators:
# - Bollinger Bands
# - Exponential Moving Average,
# - Moving Average Convergence Divergence
# - Relative Strength Index
df = calculate_technical_indicators(df)
# Calculate magnitude of close price changes
df = calculate_magnitude(df)
# Fill NaN values with None and drop rows with null values
df = df.fill_nan(None).drop_nulls()

df.head()

In [ ]:
plot_candlestick_chart(df)

In [ ]:
plot_magnitude(df, start=datetime(2026, 5, 1))

In [ ]:
df = df.filter(pl.col(DATETIME) >= datetime(2024, 8, 1))

reactive = df[MAGNITUDE]
oracle = df[MAGNITUDE].shift(-1)
returns, profits = calculate_trade_returns(
    reactive,
    oracle,
    cost_pct=0.0025,
    slippage_pct=0.001,
)

plot_trade_returns(returns, profits)